In [1]:
## Importing Necessary Libraries
import requests
import json
from datetime import datetime
from pathlib import Path
import pandas as pd

In [2]:
## Galaxy Tools API Endpoint
url = "https://usegalaxy.org/api/tools"

In [3]:
## Setting Up Directories
today = datetime.now().strftime("%Y-%m-%d")
output_dir = Path("../data/raw/usegalaxy")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / f"galaxy_tools_{today}.json"

In [4]:

response = requests.get(url, timeout=60)
response.raise_for_status()

data = response.json()

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print("Saved:", output_file)
print("Top-level records:", len(data))

Saved: ../data/raw/usegalaxy/galaxy_tools_2026-06-16.json
Top-level records: 87


In [5]:
## Parse all tools into a clean
json_file = sorted(Path("../data/raw/usegalaxy").glob("galaxy_tools_*.json"))[-1]

with open(json_file, "r", encoding="utf-8") as f:
    data = json.load(f)

In [6]:
def extract_tools(obj):
    tools = []

    if isinstance(obj, dict):
        model_class = str(obj.get("model_class", ""))

        if model_class in [
            "Tool",
            "DataSourceTool",
            "ExpressionTool",
            "BuildListCollectionTool",
            "DuplicateFileToCollectionTool",
            "ConvertSampleSheetTool",
            "FlattenTool",
            "NestTool",
            "MergeCollectionTool",
            "SplitPairedAndUnpairedTool",
            "FilterFailedDatasetsTool",
            "FilterEmptyDatasetsTool",
            "FilterNullTool",
            "KeepSuccessDatasetsTool",
            "FilterFromFileTool",
            "ZipCollectionTool",
            "UnzipCollectionTool",
            "CrossProductFlatCollectionTool",
            "CrossProductNestedCollectionTool",
            "HarmonizeTool",
            "SortTool",
            "TagFromFileTool",
            "RelabelFromFileTool",
            "ExtractDatasetCollectionTool",
            "ApplyRulesTool"
        ]:
            tools.append(obj)

        for value in obj.values():
            if isinstance(value, (dict, list)):
                tools.extend(extract_tools(value))

    elif isinstance(obj, list):
        for item in obj:
            tools.extend(extract_tools(item))

    return tools

In [7]:
tools = extract_tools(data)
df_tools = pd.json_normalize(tools)

In [8]:
df_tools["tool_text"] = (
    df_tools.get("name", "").fillna("").astype(str)
    + " "
    + df_tools.get("description", "").fillna("").astype(str)
    + " "
    + df_tools.get("panel_section_name", "").fillna("").astype(str)
)

output_file = Path("../data/processed/usegalaxy_tool_registry.csv")
output_file.parent.mkdir(parents=True, exist_ok=True)

df_tools.to_csv(output_file, index=False)

print("Total tools:", len(df_tools))
print("Saved:", output_file)
df_tools.head()

Total tools: 2321
Saved: ../data/processed/usegalaxy_tool_registry.csv


,model_class,id,name,version,description,labels,icon,edam_operations,edam_topics,hidden,...,target,has_parameters,panel_section_id,panel_section_name,form_style,tool_shed_repository.name,tool_shed_repository.owner,tool_shed_repository.changeset_revision,tool_shed_repository.tool_shed,tool_text
0,Tool,toolshed.g2.bx.psu.edu/repos/iuc/ncbi_acc_down...,NCBI Accession Download,0.2.8+galaxy0,Download sequences from GenBank/RefSeq by acce...,[],NaN,[],[],,...,galaxy_main,True,get_data,Get Data,regular,ncbi_acc_download,iuc,e063168e0a81,toolshed.g2.bx.psu.edu,NCBI Accession Download Download sequences fro...
1,Tool,toolshed.g2.bx.psu.edu/repos/galaxyp/unipept/u...,Unipept,6.2.4+galaxy1,retrieve taxonomy for peptides,[],NaN,[],[],,...,galaxy_main,True,get_data,Get Data,regular,unipept,galaxyp,4edfc76f8a29,toolshed.g2.bx.psu.edu,Unipept retrieve taxonomy for peptides Get Data
2,Tool,toolshed.g2.bx.psu.edu/repos/iuc/sra_tools/fas...,Download and Extract Reads in FASTQ,3.1.1+galaxy1,format from NCBI SRA,[],NaN,"[operation_2422, operation_0335]","[topic_0622, topic_0091]",,...,galaxy_main,True,get_data,Get Data,regular,sra_tools,iuc,8848455c0270,toolshed.g2.bx.psu.edu,Download and Extract Reads in FASTQ format fro...
3,DataSourceTool,ebi_sra_main,EBI SRA,1.0.1,ENA SRA,[],NaN,[operation_0224],[],,...,_top,True,get_data,Get Data,special,NaN,NaN,NaN,NaN,EBI SRA ENA SRA Get Data
4,Tool,toolshed.g2.bx.psu.edu/repos/iuc/sra_tools/sam...,Download and Extract Reads in BAM,3.1.1+galaxy1,format from NCBI SRA,[],NaN,"[operation_2422, operation_0335]","[topic_0622, topic_0091]",,...,galaxy_main,True,get_data,Get Data,regular,sra_tools,iuc,8848455c0270,toolshed.g2.bx.psu.edu,Download and Extract Reads in BAM format from ...
